<a href="https://colab.research.google.com/github/D3EES/FlyRank/blob/main/work/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring

**Task Type:** Ranking / Scoring

**Why:**
The goal of this task is to prioritize content pages for human editors to refresh. Since editorial resources are constrained (e.g., they can only review $K$ pages per week), we do not need to make an absolute binary decision ("will decline" or "will not decline") for all pages. Instead, we need a relative ordering of the pages based on their opportunity or risk of decline. A ranking/scoring approach allows us to assign a score to each page and sort them, presenting the highest-priority pages at the top of the queue.

In [ ]:
import pandas as pd
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")
for _ in range(3):
    if os.path.exists("data/raw/content_refresh_anonymized.csv"):
        break
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Count pages per client
pages_per_client = df.groupby("client_id")["content_id"].count()

print(f"Total pages across all clients: {len(df):,}")
print(f"Total unique clients: {df['client_id'].nunique()}")
print(f"Average pages per client: {pages_per_client.mean():.1f}")
print(f"Max pages for a single client: {pages_per_client.max()}")
print("\nTop 5 clients by content page count:")
print(pages_per_client.sort_values(ascending=False).head())

print("\nWhy Ranking/Scoring is needed:")
print("Because clients have hundreds or thousands of pages (up to "
      f"{pages_per_client.max()} pages), a simple manual review is impossible. "
      "Ranking provides a prioritized queue matching the editorial capacity.")

Working dir: /flyrank-ml-internship-starter
Starter data found. You're ready.
Total pages across all clients: 30,000
Total unique clients: 32
Average pages per client: 937.5
Max pages for a single client: 7008

Top 5 clients by content page count:
client_id
client_19581e27de    7008
client_6208ef0f77    3681
client_4e07408562    2294
client_3fdba35f04    2267
client_f369cb89fc    1796
Name: content_id, dtype: int64

Why Ranking/Scoring is needed:
Because clients have hundreds or thousands of pages (up to 7008 pages), a simple manual review is impossible. Ranking provides a prioritized queue matching the editorial capacity.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** Future content performance decline (specifically, a page's traffic or visibility dropping significantly in a future time window).

**Proxy Label:** In the starter dataset, the target is modeled using `is_declining_label`, which is defined as `trend_direction == "down"`.
- **Where it comes from:** In this starter slice, it is derived from `trend_pct` in the current 90-day window.
- **Proxy warning:** Because `trend_direction` is computed from `trend_pct`, this is a proxy label. In a true production system, using features from the same window as the label causes data leakage. For a robust ML task, we should use a future window (e.g., next 30 days) to define the target, separate from our features (prior 90 days). Here, we must ensure we exclude `trend_direction` and `trend_pct` from our features.


In [ ]:
import pandas as pd
import numpy as np
import os

# Resolve workspace directory relative to notebook
for _ in range(3):
    if os.path.exists("data/raw/content_refresh_anonymized.csv"):
        break
    os.chdir("..")

# Load the starter dataset to analyze the class distribution
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Compute the target label distribution
df["is_declining_label"] = df["trend_direction"] == "down"
class_counts = df["is_declining_label"].value_counts()
class_pcts = df["is_declining_label"].value_counts(normalize=True) * 100

print(f"Dataset size: {len(df)} rows")
print("\nTarget Label (is_declining_label) Distribution:")
for val, count in class_counts.items():
    pct = class_pcts[val]
    print(f"  {val} (Declining): {count} rows ({pct:.2f}%)")



Dataset size: 30000 rows

Target Label (is_declining_label) Distribution:
  True (Declining): 16262 rows (54.21%)
  False (Declining): 13738 rows (45.79%)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Metric:** **Precision@K** (specifically **Precision@50**)
**Secondary Metric:** **Average Precision (AP)** / **ROC-AUC**

**Defense:**
Editorial capacity is strictly limited. If the content team can only review 50 pages a week, the most critical question is: *"Of the top 50 pages we recommended for refresh, how many actually needed a refresh (were declining/had opportunity)?"* This is precisely measured by Precision@50.
- **Average Precision** evaluates the quality of the ranking across all thresholds, which is helpful to ensure the scoring is robust.
- **ROC-AUC** helps measure the model's ability to separate declining pages from stable ones across all thresholds.

**What means 'good':**
- A simple heuristic like ranking by age alone can yield a high raw Precision@50 (e.g. **0.760**), but this is a misleading/poor rule in practice: it prioritizes very old pages that have no traffic or visibility, resulting in wasted editor effort.
- When we evaluate the multi-signal `baseline_rules` (which factor in traffic, visibility, position, and freshness), the Precision@50 is **0.240**.
- Our machine learning model (`random_forest`) achieves a Precision@50 of **0.740** while properly targeting high-value visible pages.
- Therefore, a Precision@50 of **>= 0.70** on visible, high-traffic pages is considered "good" and represents a massive improvement over the baseline, saving significant editorial time.

In [ ]:
top_50_by_age = df.sort_values(by="content_age_days", ascending=False).head(50)
precision_at_50_age = top_50_by_age["is_declining_label"].mean()
print(f"Baseline Precision@50 (ranking by content age): {precision_at_50_age:.3f}")


Baseline Precision@50 (ranking by content age): 0.760


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis:** One row represents one **pseudonymized content item** (`content_id`) for a specific client (`client_id`) with aggregated search and engagement metrics over a trailing 90-day window.


In [ ]:
import pandas as pd
import os

# Resolve workspace directory relative to notebook
for _ in range(3):
    if os.path.exists("data/raw/content_refresh_anonymized.csv"):
        break
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Filter the data to represent the active slice (pages with impressions and age >= 90 days)
df_active = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()

print(f"Active Slice Shape: {df_active.shape}")
print("Confirming unique content IDs:")
is_unique = df_active["content_id"].nunique() == len(df_active)
print(f"  Is content_id unique? {is_unique} ({df_active['content_id'].nunique()} unique IDs)")

# Display a sample of the dataframe
key_cols = [
    "content_id", "client_id", "impressions_90d", "sessions_90d",
    "content_age_days", "avg_position", "ctr", "is_declining_label"
]
df_active["is_declining_label"] = df_active["trend_direction"] == "down"
df_active[key_cols].head()


Active Slice Shape: (30000, 44)
Confirming unique content IDs:
  Is content_id unique? True (30000 unique IDs)


,content_id,client_id,impressions_90d,sessions_90d,content_age_days,avg_position,ctr,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,17,187,10.6,0.76,True
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,20.3,0.05,True
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,36.5,0.09,True
3,content_331d6c4de07b,client_19581e27de,11751,78,463,6.2,0.49,False
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,44.0,0.13,True


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


**Why ML is needed:**
A single manual rule or threshold is too rigid and fails to capture the multi-dimensional nature of content decay. For example:
1. **Age alone is not enough:** A page can be old but highly stable and high-performing, or young and rapidly declining.
2. **Simple combinations fail:** If we create a rule like `age >= 180 and impressions_90d >= 500`, we end up with low precision because many old, highly-visible pages are actually healthy, and we miss younger declining pages (low recall).
3. **Complex non-linear relationships:** Performance decay is a function of search demand (impressions), engagement (sessions, scroll rate, engagement rate), search engine ranking position, and click-through rates. These signals interact in non-linear ways that a simple set of nested `if` statements cannot robustly model across 32+ different clients.

In [ ]:
df_active["rule_stale_visible"] = (df_active["content_age_days"] >= 180) & (df_active["impressions_90d"] >= 500)

rule_positives = df_active[df_active["rule_stale_visible"]]
precision_rule = rule_positives["is_declining_label"].mean()
recall_rule = rule_positives["is_declining_label"].sum() / df_active["is_declining_label"].sum()

print(f"Fixed Rule: Content Age >= 180 days and Impressions >= 500")
print(f"  Captured: {len(rule_positives)} pages")
print(f"  Precision: {precision_rule:.3f}")
print(f"  Recall: {recall_rule:.3f}")

# Let's test another rule: Low CTR Visible Page (impressions >= 500 and position <= 20 and ctr < 0.5)
df_active["rule_low_ctr"] = (df_active["impressions_90d"] >= 500) & (df_active["avg_position"] > 0) & (df_active["avg_position"] <= 20) & (df_active["ctr"] < 0.5)
rule_low_ctr_pos = df_active[df_active["rule_low_ctr"]]
precision_ctr = rule_low_ctr_pos["is_declining_label"].mean()
recall_ctr = rule_low_ctr_pos["is_declining_label"].sum() / df_active["is_declining_label"].sum()

print(f"\nFixed Rule: Low CTR Visible Page")
print(f"  Captured: {len(rule_low_ctr_pos)} pages")
print(f"  Precision: {precision_ctr:.3f}")
print(f"  Recall: {recall_ctr:.3f}")


Fixed Rule: Content Age >= 180 days and Impressions >= 500
  Captured: 9929 pages
  Precision: 0.540
  Recall: 0.330

Fixed Rule: Low CTR Visible Page
  Captured: 9759 pages
  Precision: 0.627
  Recall: 0.376


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.